In [50]:
import numpy as np
import finitewave as fw

mesh = np.ones((100, 100, 100), dtype=np.int8)
mesh += (np.random.rand(*mesh.shape) < 0.2).astype(np.int8)

tissue = fw.CardiacTissue(mesh.shape, dr=0.2)
tissue.mesh = mesh

diffusion_model = fw.DiffusionModel()
K, M = diffusion_model.compute_weights(tissue)

In [51]:
from numba import njit, prange


@njit(parallel=True, fastmath=True)
def matvec_reindex_numba(indptr, indices, data, x, out, indexes):
    """
    Computes out = A @ x for a sparse matrix A in CSR format.
    """
    n = indptr.shape[0] - 1
    for i in prange(n):
        start, end = indptr[i], indptr[i+1]
        if start == end:
            continue
        out_i = 0.
        for j in range(start, end):
            jj = indices[j]
            jj = indexes[jj]
            out_i += data[j] * x.flat[jj]

        ii = indexes[i]
        out.flat[ii] = out_i
    return out


@njit(parallel=True, fastmath=True)
def matvec_numba(indptr, indices, data, x, out, indexes):
    """
    Computes out = A @ x for a sparse matrix A in CSR format.
    """
    # n = len(indexes)
    n = indptr.shape[0] - 1
    for i in prange(n):
        # i = indexes[ii]
        start, end = indptr[i], indptr[i+1]
        if start == end:
            continue
        out_i = 0.
        for j in range(start, end):
            jj = indices[j]
            out_i += data[j] * x.flat[jj]

        out.flat[i] = out_i
    return out

In [52]:
indexes = tissue.myo_indexes
K_reindex = K[indexes, :][:, indexes].copy()
x = np.random.rand(K.shape[0])

K_reindex.indptr, K_reindex.indices, K_reindex.data

y = np.zeros_like(x)
y_indexes = matvec_reindex_numba(K_reindex.indptr, K_reindex.indices, K_reindex.data, x, y, indexes)
y = matvec_numba(K.indptr, K.indices, K.data, x, y, indexes)

In [53]:
y = np.zeros_like(x)
%timeit matvec_reindex_numba(K_reindex.indptr, K_reindex.indices, K_reindex.data, x, y, indexes)
y = np.zeros_like(x)
%timeit matvec_numba(K.indptr, K.indices, K.data, x, y, indexes)

916 μs ± 34.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
931 μs ± 17.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [46]:
@njit(parallel=True, fastmath=True)
def axpy(a, x, y):
    """
    Computes y = a * x + y for vectors x and y.
    """
    n = x.shape[0]
    for i in prange(n):
        y.flat[i] += a * x.flat[i]
    return y


@njit(parallel=True, fastmath=True)
def axpy_regular(a, x, y):
    """
    Computes y = a * x + y for vectors x and y.
    """
    return a * x + y


x = np.random.rand(1000000)
y = np.random.rand(1000000)
a = 2.0

axpy(a, x, y)
axpy_regular(a, x, y)

%timeit axpy(a, x, y)
%timeit axpy_regular(a, x, y)

76.4 μs ± 1.9 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
287 μs ± 1.35 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [54]:
from jax import numpy as jnp
import jax
from jax.experimental import sparse as jsp


@jax.jit
def jax_matvec(jsp_matrix, x):
    return jsp_matrix @ x

K_jsp = jsp.BCSR.from_scipy_sparse(K)
x = jnp.array(np.random.rand(K.shape[0]))
jax_matvec(K_jsp, x)

%timeit jax_matvec(K_jsp, x)


4.79 ms ± 83.2 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [55]:
def scipy_matvec(scipy_matrix, x):
    return scipy_matrix @ x

x = np.random.rand(K.shape[0])
scipy_matvec(K, x)
%timeit scipy_matvec(K, x)

5.28 ms ± 40.8 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
